# pre_processing_upgrade_3 — Kaggle runner

Universal analyzer-agnostic video preprocessing (A1) + task-importance mask (A2) + real-codec calibration (A3), in front of a **frozen** x264/x265.

**Before Run All:** Accelerator = GPU T4/P100, Internet = On, and add the `rohanmallick/kinetics-train-5per` dataset as input. See `docs/KAGGLE.md`.

## 0. Clone + install (torch/torchvision/ffmpeg already on Kaggle)

In [ ]:
!git clone https://github.com/wagur1/pre_processing_upgrade_3.git
%cd pre_processing_upgrade_3
!pip -q install compressai av
!ffmpeg -hide_banner -encoders | grep -E 'libx264|libx265'

## 1. Self-checks (no dataset/GPU needed)

In [ ]:
!python -m src.models.virtual_codec
!python -m src.models.task_mask
!python -m src.tasks.multi_teacher
!python -m src.metrics.bd_rate
!python -m src.models.ste_codec

## 2. Build a <=3GB balanced Kinetics index

In [ ]:
!python -m src.data.prepare_3gb \
    --root /kaggle/input/kinetics-train-5per/train \
    --out data/index/kinetics_3gb.json \
    --cap-gb 3.0 --val-frac 0.1 --test-frac 0.1 --backbone r3d_18

## 3. STAGE 1 — proxy pretrain (A1 panel + A2 mask + A3 anneal)

In [ ]:
!python train.py --config configs/universal_action_recognition.yaml \
    data.index=data/index/kinetics_3gb.json train.epochs=5 train.batch_size=4

## 4. STAGE 2 — real-codec (STE) calibration fine-tune (A3)

In [ ]:
!python train.py --config configs/universal_action_recognition.yaml \
    data.index=data/index/kinetics_3gb.json \
    codec.kind=ste codec.ste_codec=h265 \
    train.finetune=true train.resume=false train.epochs=6 train.lr=3e-5 train.max_steps=400

## 5. Evaluate on HELD-OUT analyzer (r2plus1d_18) + real x264/x265

In [ ]:
!python evaluate.py --config configs/universal_action_recognition.yaml \
    --ckpt outputs/checkpoints/preprocessor.pth \
    data.index=data/index/kinetics_3gb.json eval.held_out_backbone=r2plus1d_18 eval.per_sequence=true

In [ ]:
import json
res = json.load(open('outputs/eval/results.json'))
print('bd_prep_gain (same-codec, THE claim):')
for k, v in res.get('bd_prep_gain', {}).items():
    rate, acc = v.get('bd_rate_pct'), v.get('bd_accuracy')
    rs = f'{rate:+.2f}%' if rate is not None else 'undefined'
    acs = f'{acc:+.4f}' if acc is not None else 'undefined'
    print(f'  {k:28s} BD-Rate {rs:>10s}  BD-Acc {acs}')
print('per-sequence files: outputs/eval/sequence_points.csv, sequence_bd_rate.csv, sequence_bd_rate.json')

## 6. (optional) multi-seed 95% CI

In [ ]:
!bash kaggle/run.sh   # or loop seeds as in docs/KAGGLE.md section 5, then:
# !python kaggle/report_ci.py outputs/seed_*/eval/results.json